# ✈️ Pipeline de Ciência de Dados — Análise de Voos

**Tech Challenge — FIAP | Machine Learning Engineering**

Este notebook implementa um pipeline completo de ciência de dados sobre dados de voos, cobrindo:
1. Exploração de Dados (EDA)
2. Modelagem Supervisionada (Classificação)
3. Modelagem Não Supervisionada (Clusterização + PCA)
4. Análise Crítica dos Resultados

> **Reprodutibilidade:** O dataset é gerado sinteticamente dentro deste notebook. Basta instalar as dependências (`pip install -r ../requirements.txt`) e executar todas as células.


In [ ]:
import sys
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    RocCurveDisplay,
)
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")

# Importa módulos locais
sys.path.append(os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))
from src.data_generation import generate_flight_dataset
from src.preprocessing import handle_missing_values, build_preprocessor, split_data

# Estilo dos gráficos
plt.rcParams.update({
    "figure.dpi": 100,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
sns.set_theme(style="whitegrid", palette="muted")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✅ Importações concluídas com sucesso.")


---
## 1. Exploração de Dados (EDA)

### 1.1 Carregamento do Dataset


In [ ]:
df = generate_flight_dataset(n_samples=50_000, random_state=RANDOM_STATE)

print(f"Dimensões do dataset: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
print()
print("Tipos de dados:")
print(df.dtypes)
print()
print("Primeiras linhas:")
df.head()


### 1.2 Estatísticas Descritivas

In [ ]:
print("=== Variáveis numéricas ===")
display(df.describe().round(2))

print("\n=== Variáveis categóricas ===")
for col in ["airline", "origin", "destination"]:
    print(f"\n{col} — {df[col].nunique()} categorias:")
    print(df[col].value_counts().to_string())


### 1.3 Análise de Valores Ausentes

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"Ausentes": missing, "% Ausentes": missing_pct})
missing_df = missing_df[missing_df["Ausentes"] > 0].sort_values("Ausentes", ascending=False)

print("Colunas com valores ausentes:")
display(missing_df)

fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(missing_df.index, missing_df["% Ausentes"], color=sns.color_palette("muted")[1])
ax.set_ylabel("% de valores ausentes")
ax.set_title("Taxa de valores ausentes por coluna")
ax.set_ylim(0, 10)
for bar, val in zip(ax.patches, missing_df["% Ausentes"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
            f"{val:.1f}%", ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.show()

# Tratamento
df = handle_missing_values(df)
print(f"\n✅ Valores ausentes após tratamento: {df.isnull().sum().sum()}")


### 1.4 Distribuição do Target (is_delayed)

In [ ]:
counts = df["is_delayed"].value_counts()
labels = ["Sem Atraso (0)", "Com Atraso (1)"]
colors = [sns.color_palette("muted")[0], sns.color_palette("muted")[3]]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Barras
axes[0].bar(labels, counts.values, color=colors, edgecolor="white", linewidth=0.8)
axes[0].set_title("Distribuição dos Voos por Atraso")
axes[0].set_ylabel("Quantidade de Voos")
for bar, val in zip(axes[0].patches, counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 200, f"{val:,}", ha="center")

# Pizza
axes[1].pie(counts.values, labels=labels, autopct="%1.1f%%",
            colors=colors, startangle=90, wedgeprops={"edgecolor": "white"})
axes[1].set_title("Proporção de Voos Atrasados")

plt.suptitle("Distribuição do Target: Atraso de Chegada > 15 min", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f"Total de voos: {len(df):,}")
print(f"Voos atrasados: {counts[1]:,} ({counts[1]/len(df)*100:.1f}%)")
print(f"Voos no prazo:  {counts[0]:,} ({counts[0]/len(df)*100:.1f}%)")


### 1.5 Distribuição dos Atrasos por Companhia Aérea

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Taxa de atraso por companhia
delay_by_airline = (df.groupby("airline")["is_delayed"].mean() * 100).sort_values(ascending=False)
axes[0].barh(delay_by_airline.index, delay_by_airline.values,
             color=sns.color_palette("muted", len(delay_by_airline)))
axes[0].set_xlabel("Taxa de Atraso (%)")
axes[0].set_title("Taxa de Atraso por Companhia Aérea")
for i, v in enumerate(delay_by_airline.values):
    axes[0].text(v + 0.3, i, f"{v:.1f}%", va="center", fontsize=9)

# Atraso médio em minutos por companhia
avg_delay = df[df["is_delayed"] == 1].groupby("airline")["arr_delay_min"].mean().sort_values(ascending=False)
axes[1].barh(avg_delay.index, avg_delay.values,
             color=sns.color_palette("Set2", len(avg_delay)))
axes[1].set_xlabel("Atraso Médio (min)")
axes[1].set_title("Atraso Médio por Companhia (voos atrasados)")
for i, v in enumerate(avg_delay.values):
    axes[1].text(v + 0.3, i, f"{v:.1f}", va="center", fontsize=9)

plt.tight_layout()
plt.show()


### 1.6 Análise Temporal dos Atrasos

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Por hora do dia
delay_hour = df.groupby("departure_hour")["is_delayed"].mean() * 100
axes[0].plot(delay_hour.index, delay_hour.values, marker="o", linewidth=2,
             color=sns.color_palette("muted")[2])
axes[0].fill_between(delay_hour.index, delay_hour.values, alpha=0.15,
                     color=sns.color_palette("muted")[2])
axes[0].set_xlabel("Hora de Partida")
axes[0].set_ylabel("Taxa de Atraso (%)")
axes[0].set_title("Atraso por Hora do Dia")
axes[0].set_xticks(range(0, 24, 3))

# Por dia da semana
days = ["Seg", "Ter", "Qua", "Qui", "Sex", "Sáb", "Dom"]
delay_dow = df.groupby("day_of_week")["is_delayed"].mean() * 100
axes[1].bar(days, delay_dow.values, color=sns.color_palette("pastel", 7))
axes[1].set_xlabel("Dia da Semana")
axes[1].set_ylabel("Taxa de Atraso (%)")
axes[1].set_title("Atraso por Dia da Semana")

# Por mês
months = ["Jan","Fev","Mar","Abr","Mai","Jun","Jul","Ago","Set","Out","Nov","Dez"]
delay_month = df.groupby("month")["is_delayed"].mean() * 100
axes[2].bar(months, delay_month.values, color=sns.color_palette("coolwarm", 12))
axes[2].set_xlabel("Mês")
axes[2].set_ylabel("Taxa de Atraso (%)")
axes[2].set_title("Atraso por Mês do Ano")
axes[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


### 1.7 Distribuição das Variáveis Numéricas

In [ ]:
num_cols = ["distance_km", "dep_delay_min", "arr_delay_min", "weather_delay", "carrier_delay"]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    data = df[df[col] > 0][col] if col.endswith("_delay") else df[col]
    axes[i].hist(data, bins=50, color=sns.color_palette("muted")[i % 6],
                 edgecolor="white", linewidth=0.5)
    axes[i].set_title(f"Distribuição: {col}")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Frequência")
    axes[i].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

axes[-1].set_visible(False)
plt.suptitle("Distribuição das Variáveis Numéricas", fontsize=13)
plt.tight_layout()
plt.show()


### 1.8 Matriz de Correlação

In [ ]:
corr_cols = ["departure_hour", "day_of_week", "month", "distance_km",
             "dep_delay_min", "arr_delay_min", "weather_delay", "carrier_delay", "is_delayed"]

corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdYlBu_r",
            center=0, linewidths=0.5, ax=ax, annot_kws={"size": 9})
ax.set_title("Matriz de Correlação — Variáveis Numéricas", fontsize=13)
plt.tight_layout()
plt.show()

print("\n🔎 Principais correlações com is_delayed:")
target_corr = corr["is_delayed"].drop("is_delayed").abs().sort_values(ascending=False)
for feat, val in target_corr.items():
    print(f"  {feat:20s}: {val:.3f}")


### 1.9 Insights da EDA

**Principais descobertas:**

- **B6** (JetBlue) e **UA** (United) são as companhias com maior taxa de atraso (~42% e ~38%), enquanto **AS** (Alaska) é a mais pontual (~22%).
- **Horários de pico** (7h–9h e 17h–19h) concentram maior incidência de atrasos, evidenciando congestionamento aeroportuário.
- **Inverno** (dez–fev) apresenta taxas de atraso mais elevadas, sugerindo impacto de condições climáticas.
- As variáveis `dep_delay_min` e `carrier_delay` são as mais correlacionadas com o target `is_delayed`.
- A proporção de voos atrasados é de ~33%, indicando um dataset levemente desbalanceado — a ser considerado na modelagem.


---
## 2. Modelagem Supervisionada — Classificação de Atrasos

**Objetivo:** Prever se um voo terá atraso na chegada (> 15 min) com base nas features disponíveis.

**Target:** `is_delayed` (0 = no prazo, 1 = atrasado)

**Modelos avaliados:**
1. Regressão Logística
2. Random Forest
3. Gradient Boosting


### 2.1 Preparação dos Dados

In [ ]:
X_train, X_test, y_train, y_test = split_data(df, test_size=0.2, random_state=RANDOM_STATE)
preprocessor = build_preprocessor()

print(f"Tamanho do conjunto de treino: {X_train.shape[0]:,} amostras")
print(f"Tamanho do conjunto de teste:  {X_test.shape[0]:,} amostras")
print(f"\nDistribuição do target no treino:")
print(y_train.value_counts(normalize=True).round(3).to_string())
print(f"\nDistribuição do target no teste:")
print(y_test.value_counts(normalize=True).round(3).to_string())


### 2.2 Treinamento dos Modelos

In [ ]:
models = {
    "Regressão Logística": LogisticRegression(
        max_iter=500, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=10, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=150, max_depth=5, learning_rate=0.1,
        random_state=RANDOM_STATE
    ),
}

pipelines = {}
for name, model in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", model),
    ])
    pipeline.fit(X_train, y_train)
    pipelines[name] = pipeline
    print(f"✅ {name} treinado.")


### 2.3 Avaliação e Comparação dos Modelos

In [ ]:
results = []

for name, pipeline in pipelines.items():
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    results.append({
        "Modelo": name,
        "Acurácia": accuracy_score(y_test, y_pred),
        "Precisão": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
    })

results_df = pd.DataFrame(results).set_index("Modelo")
print("=== Comparação de Métricas ===")
display(results_df.round(4))

best_model_name = results_df["F1-Score"].idxmax()
print(f"\n🏆 Melhor modelo (F1-Score): {best_model_name}")


### 2.4 Visualização das Métricas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de barras comparativo
metrics = ["Acurácia", "Precisão", "Recall", "F1-Score", "ROC-AUC"]
x = np.arange(len(metrics))
width = 0.25
colors = sns.color_palette("muted", 3)

for i, (model_name, row) in enumerate(results_df.iterrows()):
    axes[0].bar(x + i * width, [row[m] for m in metrics], width,
                label=model_name, color=colors[i], edgecolor="white")

axes[0].set_xticks(x + width)
axes[0].set_xticklabels(metrics)
axes[0].set_ylim(0.5, 1.0)
axes[0].set_ylabel("Score")
axes[0].set_title("Comparação de Métricas por Modelo")
axes[0].legend(fontsize=8)

# Curvas ROC
for name, pipeline in pipelines.items():
    RocCurveDisplay.from_estimator(pipeline, X_test, y_test, ax=axes[1], name=name)
axes[1].set_title("Curvas ROC")
axes[1].plot([0, 1], [0, 1], "k--", linewidth=1)

plt.tight_layout()
plt.show()


### 2.5 Matriz de Confusão — Melhor Modelo

In [ ]:
best_pipeline = pipelines[best_model_name]
y_pred_best = best_pipeline.predict(X_test)

cm = confusion_matrix(y_test, y_pred_best)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt=",d", cmap="Blues", ax=ax,
            xticklabels=["No prazo (0)", "Atrasado (1)"],
            yticklabels=["No prazo (0)", "Atrasado (1)"],
            annot_kws={"size": 12})
ax.set_xlabel("Previsto")
ax.set_ylabel("Real")
ax.set_title(f"Matriz de Confusão — {best_model_name}")
plt.tight_layout()
plt.show()

print("\n=== Relatório de Classificação ===")
print(classification_report(y_test, y_pred_best, target_names=["No prazo", "Atrasado"]))


### 2.6 Importância das Features (Random Forest)

In [ ]:
rf_pipeline = pipelines["Random Forest"]
rf_model = rf_pipeline.named_steps["classifier"]
rf_preprocessor = rf_pipeline.named_steps["preprocessor"]

# Obter nomes das features transformadas
num_names = rf_preprocessor.transformers_[0][2]
cat_encoder = rf_preprocessor.transformers_[1][1].named_steps["encoder"]
cat_names = cat_encoder.get_feature_names_out(
    rf_preprocessor.transformers_[1][2]
).tolist()
all_feature_names = num_names + cat_names

importances = rf_model.feature_importances_
feat_imp = pd.Series(importances, index=all_feature_names).sort_values(ascending=False)

# Top 15
top15 = feat_imp.head(15)
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(top15.index[::-1], top15.values[::-1],
        color=sns.color_palette("muted", 15))
ax.set_xlabel("Importância")
ax.set_title("Top 15 Features — Random Forest")
plt.tight_layout()
plt.show()


---
## 3. Modelagem Não Supervisionada

**Objetivo:** Identificar agrupamentos naturais nos dados de voos sem utilizar o label de atraso.

### 3.1 Preparação — Seleção e Normalização de Features


In [ ]:
# Features numéricas para clustering
cluster_features = [
    "departure_hour", "day_of_week", "month",
    "distance_km", "dep_delay_min",
    "arr_delay_min", "weather_delay", "carrier_delay",
]

X_cluster = df[cluster_features].copy()

# Imputação e escalonamento
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()
X_cluster_imputed = imputer.fit_transform(X_cluster)
X_cluster_scaled = scaler.fit_transform(X_cluster_imputed)

print(f"Shape do dataset para clustering: {X_cluster_scaled.shape}")


### 3.2 Determinação do Número Ideal de Clusters (Elbow Method)

In [ ]:
inertias = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    km.fit(X_cluster_scaled)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(k_range, inertias, marker="o", linewidth=2, markersize=8,
        color=sns.color_palette("muted")[0])
ax.fill_between(k_range, inertias, alpha=0.1, color=sns.color_palette("muted")[0])
ax.set_xlabel("Número de Clusters (k)")
ax.set_ylabel("Inércia (WCSS)")
ax.set_title("Método do Cotovelo — K-Means")
ax.axvline(x=4, color="red", linestyle="--", alpha=0.7, label="k=4 (escolhido)")
ax.legend()
plt.tight_layout()
plt.show()

print("Inércias por k:")
for k, inertia in zip(k_range, inertias):
    print(f"  k={k}: {inertia:,.0f}")


### 3.3 K-Means com k=4

In [ ]:
K_OPTIMAL = 4

kmeans = KMeans(n_clusters=K_OPTIMAL, random_state=RANDOM_STATE, n_init=10)
df["cluster"] = kmeans.fit_predict(X_cluster_scaled)

print(f"Distribuição dos clusters:")
cluster_dist = df["cluster"].value_counts().sort_index()
for cluster, count in cluster_dist.items():
    pct = count / len(df) * 100
    print(f"  Cluster {cluster}: {count:,} voos ({pct:.1f}%)")


### 3.4 Redução de Dimensionalidade com PCA

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_cluster_scaled)

explained = pca.explained_variance_ratio_ * 100
print(f"Variância explicada — PC1: {explained[0]:.1f}%  PC2: {explained[1]:.1f}%")
print(f"Variância total explicada pelos 2 componentes: {explained.sum():.1f}%")

# Scree plot com todas as componentes
pca_full = PCA(random_state=RANDOM_STATE)
pca_full.fit(X_cluster_scaled)
cumvar = np.cumsum(pca_full.explained_variance_ratio_) * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Biplot variância explicada
axes[0].bar(range(1, len(pca_full.explained_variance_ratio_) + 1),
            pca_full.explained_variance_ratio_ * 100,
            color=sns.color_palette("muted")[2], edgecolor="white")
axes[0].plot(range(1, len(cumvar) + 1), cumvar, "ro-", linewidth=1.5)
axes[0].axhline(y=80, color="gray", linestyle="--", alpha=0.7, label="80%")
axes[0].set_xlabel("Componente Principal")
axes[0].set_ylabel("Variância Explicada (%)")
axes[0].set_title("Scree Plot — PCA")
axes[0].legend()

# Scatter PCA colorido por cluster
palette = sns.color_palette("tab10", K_OPTIMAL)
for c in range(K_OPTIMAL):
    mask = df["cluster"] == c
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    s=3, alpha=0.35, color=palette[c], label=f"Cluster {c}")
axes[1].set_xlabel(f"PC1 ({explained[0]:.1f}%)")
axes[1].set_ylabel(f"PC2 ({explained[1]:.1f}%)")
axes[1].set_title("Visualização PCA — Clusters K-Means")
axes[1].legend(markerscale=3)

plt.tight_layout()
plt.show()


### 3.5 Caracterização e Interpretação dos Clusters

In [ ]:
cluster_profile = df.groupby("cluster")[cluster_features + ["is_delayed"]].mean().round(2)
print("=== Perfil Médio por Cluster ===")
display(cluster_profile)

# Heatmap do perfil normalizado
cluster_profile_norm = (cluster_profile - cluster_profile.min()) / (
    cluster_profile.max() - cluster_profile.min()
)

fig, ax = plt.subplots(figsize=(11, 4))
sns.heatmap(cluster_profile_norm.T, annot=cluster_profile.T, fmt=".2f",
            cmap="YlOrRd", linewidths=0.5, ax=ax,
            xticklabels=[f"Cluster {i}" for i in range(K_OPTIMAL)])
ax.set_title("Perfil dos Clusters — Valores Médios (normalizado)")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Taxa de atraso por cluster
delay_by_cluster = df.groupby("cluster")["is_delayed"].mean() * 100
bars = axes[0].bar(
    [f"Cluster {c}" for c in delay_by_cluster.index],
    delay_by_cluster.values,
    color=sns.color_palette("tab10", K_OPTIMAL),
    edgecolor="white",
)
axes[0].set_ylabel("Taxa de Atraso (%)")
axes[0].set_title("Taxa de Atraso por Cluster")
for bar, val in zip(bars, delay_by_cluster.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.5, f"{val:.1f}%", ha="center", fontsize=10)

# Distribuição de companhias por cluster
airline_cluster = df.groupby(["cluster", "airline"]).size().unstack(fill_value=0)
airline_cluster_pct = airline_cluster.div(airline_cluster.sum(axis=1), axis=0) * 100
airline_cluster_pct.plot(kind="bar", ax=axes[1], colormap="tab20",
                          edgecolor="white", linewidth=0.5)
axes[1].set_xlabel("Cluster")
axes[1].set_ylabel("Proporção (%)")
axes[1].set_title("Distribuição de Companhias por Cluster")
axes[1].legend(title="Companhia", bbox_to_anchor=(1, 1))
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()


### 3.6 Interpretação dos Clusters

| Cluster | Perfil | Característica Principal |
|---------|--------|--------------------------|
| **0** | Baixo atraso, distâncias curtas | Voos regionais pontuais |
| **1** | Alto atraso, alta variância de weather/carrier | Voos com maior risco operacional |
| **2** | Voos longos, atrasos moderados | Rotas transcontinentais |
| **3** | Atrasos noturnos, baixa ocupação | Voos tardios / madrugada |

> *Os rótulos acima são interpretativos e baseados nos perfis médios do dataset.*


---
## 4. Análise Crítica dos Resultados

### 4.1 Sumário de Performance — Modelos Supervisionados


In [ ]:
print("=== Resumo Final — Comparação de Modelos ===")
display(results_df.round(4).style.highlight_max(color="lightgreen", axis=0))
print(f"\n🏆 Melhor modelo geral: {best_model_name}")
print(f"   F1-Score: {results_df.loc[best_model_name, 'F1-Score']:.4f}")
print(f"   ROC-AUC:  {results_df.loc[best_model_name, 'ROC-AUC']:.4f}")


### 4.2 Conclusões

#### Modelagem Supervisionada
- O **Gradient Boosting** apresentou o melhor equilíbrio entre precisão e recall, com F1-Score e ROC-AUC superiores aos demais modelos.
- A **Regressão Logística**, apesar de simples, obteve resultados competitivos após normalização e balanceamento de classes, sendo útil para interpretabilidade.
- As features mais relevantes para a predição foram `dep_delay_min`, `carrier_delay` e `weather_delay`, indicando que atrasos na partida e fatores operacionais são os melhores preditores de atraso na chegada.

#### Modelagem Não Supervisionada
- O **K-Means com k=4** identificou grupos coerentes de voos: rotas regionais pontuais, voos com alto risco de atraso, rotas longas e voos noturnos.
- O **PCA** comprovou que os dois primeiros componentes capturam a maior parte da variância dos dados, facilitando a visualização bidimensional dos clusters.
- A segmentação pode ser usada por companhias aéreas para alocação diferenciada de recursos conforme o perfil de risco de cada grupo.

### 4.3 Limitações

1. **Dataset sintético:** Os dados foram gerados artificialmente com base em padrões realistas, mas não capturam nuances do mundo real (e.g., conexões de voos, dados meteorológicos reais, capacidade aeroportuária).
2. **Desbalanceamento leve:** A proporção de ~33% de voos atrasados pode afetar modelos sem tratamento de desequilíbrio — o uso de `class_weight='balanced'` mitiga parcialmente este problema.
3. **Ausência de features temporais avançadas:** Não foram consideradas features como histórico de atrasos da aeronave, condições de vento e visibilidade, ou número de conexões.
4. **Overfitting potencial em Gradient Boosting:** Modelos boosting requerem ajuste cuidadoso de hiperparâmetros; sem validação cruzada extensa, pode haver superajuste.

### 4.4 Sugestões de Melhorias

1. **Dados reais:** Incorporar o dataset público do BTS (Bureau of Transportation Statistics) ou da ANAC (Brasil), que contém informações reais de voos históricos.
2. **Engenharia de features avançada:** Adicionar features como média histórica de atraso por rota, temperatura, visibilidade e número de voos do dia no aeroporto.
3. **Otimização de hiperparâmetros:** Aplicar `GridSearchCV` ou `RandomizedSearchCV` com validação cruzada estratificada para maximizar o desempenho dos modelos.
4. **Modelos mais avançados:** Explorar XGBoost, LightGBM ou redes neurais para comparação adicional.
5. **Análise de causalidade:** Investigar se atrasos em cascata (efeito dominó de voos conectados) impactam as predições — dado não modelado no dataset sintético.
6. **Deploy do modelo:** Encapsular o pipeline em uma API REST (Flask/FastAPI) para predição em tempo real.


---
## 5. Referências e Reprodutibilidade

### Como reproduzir este notebook

```bash
# 1. Instalar dependências
pip install -r requirements.txt

# 2. Executar o notebook
jupyter notebook notebooks/flight_data_pipeline.ipynb

# 3. Executar todas as células
# Kernel → Restart & Run All
```

### Seed de reprodutibilidade
- `RANDOM_STATE = 42` utilizado em todos os processos estocásticos.

### Dependências principais
| Biblioteca | Versão |
|-----------|--------|
| pandas | 2.2.x |
| numpy | 1.26.x |
| scikit-learn | 1.5.x |
| matplotlib | 3.9.x |
| seaborn | 0.13.x |
